In [ ]:
import torch
import numpy as np
import cv2
from sklearn.datasets import load_wine
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from torch import Tensor, nn
from torch.nn import Sequential, Linear, ReLU, CrossEntropyLoss, Module, Dropout, Flatten, Conv2d, MaxPool2d, AdaptiveAvgPool2d
from torch.utils.data import DataLoader, Dataset
from torch.optim import Adam
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
class MyDataset(Dataset):
    def __init__(self, datas: np.ndarray, labels: np.ndarray) -> None:
        self._x = torch.from_numpy(datas).float()
        self._y = torch.from_numpy(labels).long()

    def __len__(self) -> int:
        return len(self._x)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        int(idx)
        return self._x[idx], self._y[idx]

In [ ]:
# class CustomModel(Module):
#     def __init__(self, input_size: int, output_size: int) -> None:
#         super().__init__()

#         # Carrega a arquitetura MobileNetV2 sem pesos pré-treinados
#         mobilenet = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)

#         # Ajusta a primeira convolução para imagens em escala de cinza
#         mobilenet.features[0][0] = __import__("torch").nn.Conv2d(
#             in_channels=1,
#             out_channels=32,
#             kernel_size=3,
#             stride=2,
#             padding=1,
#             bias=False,
#         )

#         # Substitui o classificador original pelo classificador do projeto
#         mobilenet.classifier = nn.Sequential(
#             nn.Dropout(p=0.25),
#             nn.Linear(1280, 1024),
#             nn.ReLU(),
#             nn.Dropout(p=0.25),
#             nn.Linear(1024, 512),
#             nn.ReLU(),
#             nn.Dropout(p=0.25),
#             nn.Linear(512, output_size),
#         )

#         self.net = mobilenet

#     def forward(self, x: Tensor) -> Tensor:
#         return self.net(x)

class CustomModel(Module):
    def __init__(self, input_size: int, output_size: int) -> None:
        super().__init__()

        self.net = Sequential(
            Conv2d(3, 16, kernel_size=3, padding=1),
            ReLU(),
            MaxPool2d(kernel_size=2),
            Conv2d(16, 32, kernel_size=3, padding=1),
            ReLU(),
            MaxPool2d(kernel_size=2),
            Conv2d(32, 64, kernel_size=3, padding=1),
            ReLU(),
            MaxPool2d(kernel_size=2),
            AdaptiveAvgPool2d(1),
            Flatten(),
            Dropout(p=0.25),
            Linear(64, output_size),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)

In [ ]:
def train_model_class(features, targets, kf, batch, epochs, criterion, exp=False):
    print(torch.cuda.is_available())
    print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    acc_scores = []
    f1_scores = []
    folds_loss = []
    folds_acc = []
    num_classes = len(np.unique(targets))
    for i, (train_index, test_index) in enumerate(kf.split(features, targets)):
        print(f"Treinando modelo ----- Fold {i + 1}")
        train_data = MyDataset(features[train_index], targets[train_index])
        train_loader = DataLoader(train_data, batch_size=batch, shuffle=True)
        model = CustomModel(16900, num_classes).to(device)
        optmizer = Adam(model.parameters(), lr=0.001)
        epoch_loss = []
        epoch_acc = []
        for i_epochs in range(epochs):
            model.train()
            batches_loss = []
            batches_acc = []
            for x, y in train_loader:
                x = x.float().to(device)
                y = y.to(device)
                optmizer.zero_grad()
                output = model(x)
                loss = criterion(output, y)
                loss.backward()
                optmizer.step()
                batches_loss.append(loss.item())
                batches_acc.append(accuracy_score(y.cpu().numpy(), torch.argmax(output, dim=1).cpu().numpy()))

            epoch_loss.append(np.mean(batches_loss))
            epoch_acc.append(np.mean(batches_acc))

        folds_loss.append(epoch_loss)
        folds_acc.append(epoch_acc)
        model.eval()
        with torch.no_grad():
            x_test_tensor = Tensor(features[test_index]).float().to(device)
            y_true = targets[test_index]
            output = model(x_test_tensor)
            result = torch.argmax(output, dim=1).cpu().numpy()
            acc_s = accuracy_score(y_true, result)
            f1_s = f1_score(y_true, result, average="macro")
            cm = confusion_matrix(y_true, result)
            acc_scores.append(acc_s)
            f1_scores.append(f1_s)
            print(f"Acc: {acc_s}")
            print(f"F1-Score: {f1_s}")
            print(f"CM: {cm}")

        if exp:
            break

    print(f"\nResultado modelo ----- Acc: {np.mean(acc_scores)}, F1-Score: {np.mean(f1_scores)}")

    return folds_loss, folds_acc

In [ ]:
def plot_loss(folds_loss, folds_acc, max_epochs, batch):
    folds_acc = np.array(folds_acc) * 100
    for idx, fold_l in enumerate(folds_loss):
        fig, ax1 = plt.subplots(figsize=(7, 3))
        x = np.linspace(0, len(fold_l), len(fold_l))
        ax1.plot(x, fold_l, color="red", label="Erro")
        ax1.set_xlabel("Épocas")
        ax1.set_ylabel("Erro")
        ax1.set_title(f"Épocas: {max_epochs} | Batch: {batch}")
        # ax1.set_ylim(0, 0.2)
        ax2 = ax1.twinx()
        ax2.plot(x, folds_acc[idx], color="blue", label="Acc")
        ax2.set_ylabel("Acurácia")
        # ax2.set_ylim(0.8, 1.1)
        ax2.grid()
        fig.legend()
        fig.show()

In [ ]:
def load_data(path: str, limit=1000, size=(128, 128)):
    input_dir = Path(path)
    data_array = []

    image_paths = sorted(
        p for p in input_dir.iterdir() if p.is_file()
    )[:limit]

    for image_path in image_paths:
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)

        if image is None:
            print(f"Não foi possível carregar: {image_path.name}")
            continue

        # O OpenCV lê em BGR; converte para RGB
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, size)
        data_array.append(image)

    return np.asarray(data_array, dtype=np.uint8)

In [ ]:
# Pasta com as subpastas kidney_normal e kidney_tumor
DATA_DIR = Path("/app/data/Kidney_Cancer")

class_normal = load_data(DATA_DIR / "kidney_normal")
class_tumor = load_data(DATA_DIR / "kidney_tumor")

normal_target = np.zeros(len(class_normal), dtype=int)
tumor_target = np.full(len(class_tumor), 1, dtype=int)

features = np.concatenate((class_normal, class_tumor))
features = features.transpose(0, 3, 1, 2)  # (N, 128, 128, 3) -> (N, 3, 128, 128)

targets = np.concatenate((normal_target, tumor_target))

In [ ]:
print(targets)
print(features.shape)

In [ ]:
criterion = CrossEntropyLoss()
max_epochs = 1500
batch = 64
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
folds_loss, folds_acc = train_model_class(features, targets, kf, batch, max_epochs, criterion, True)

In [ ]:
plot_loss(folds_loss, folds_acc, max_epochs, batch)